In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/content/Telecom-Customer-Churn - Telecom-Customer-Churn.csv")

In [3]:
print("Shape:", df.shape)
print(df.head())
print(df.info())
print(df.describe())

Shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies   

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

In [ ]:
df = df.drop_duplicates()

In [6]:
categorical_columns = df.select_dtypes(
    include="object"
).columns

In [7]:
for column in categorical_columns:
    df[column] = df[column].astype(str).str.strip()

In [8]:
df["customerID"] = df["customerID"].astype(str)

In [9]:
df["SeniorCitizen"] = df["SeniorCitizen"].astype(int)
df["tenure"] = df["tenure"].astype(int)

In [10]:
df["MonthlyCharges"] = pd.to_numeric(
    df["MonthlyCharges"],
    errors="coerce"
)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [11]:
numeric_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [12]:
for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

In [14]:
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

In [15]:
df[column] = df[column].clip(lower, upper)

In [16]:
df["AverageMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

In [17]:
df["TotalServices"] = (
    (df["PhoneService"] == "Yes").astype(int) +
    (df["OnlineSecurity"] == "Yes").astype(int) +
    (df["OnlineBackup"] == "Yes").astype(int) +
    (df["DeviceProtection"] == "Yes").astype(int) +
    (df["TechSupport"] == "Yes").astype(int) +
    (df["StreamingTV"] == "Yes").astype(int) +
    (df["StreamingMovies"] == "Yes").astype(int)
)

In [18]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [19]:
X = df.drop(
    columns=["customerID", "Churn"]
)

y = df["Churn"]

In [20]:
X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [22]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [23]:
print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (5634, 32)
Testing Data: (1409, 32)


In [24]:
df.to_csv(
    "Telecom_Customer_Churn_cleaned.csv",
    index=False
)

In [25]:
print("\nCleaned dataset exported successfully.")
print("Data Cleaning and Preparation Completed Successfully.")


Cleaned dataset exported successfully.
Data Cleaning and Preparation Completed Successfully.
